# Notebook 02 — Sentiment Analysis
## market-pulse-nn | Stage 2 of 5

This notebook converts raw text data into **daily sentiment scores per ticker**
that will serve as features alongside technical indicators in Notebook 03.

**Processing strategy by source:**

| Source | Articles | Method | Reason |
|---|---|---|---|
| Alpha Vantage | 824 | Use pre-computed `av_sentiment_score` directly | Already scored by AV's financial NLP model |
| RSS News | 77 | FinBERT on CPU (VADER fallback) | General market news, needs scoring |
| NewsData.io | 200 | FinBERT on CPU (VADER fallback) | Ticker-specific but 2026 dates — scored for completeness |

**Output:** `data/processed/sentiment_daily.parquet`
One row per (date, ticker) with aggregated sentiment features ready for Notebook 03.

> Run all cells top to bottom. FinBERT on CPU takes ~3-5 minutes for 277 articles.
> If it is too slow, set `USE_VADER = True` in Cell 03 to use the fast rule-based fallback.

In [ ]:
# =============================================================================
# CELL 01 | Environment Setup
#
# Same setup pattern as Notebook 01:
# - Project root resolution for VS Code and Colab
# - CFG loaded as single config source
# - Consistent plot style across all notebooks
# =============================================================================

import sys, os, json, warnings, time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

# ── Project root resolution ───────────────────────────────────────────────────
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent
    if not (PROJECT_ROOT / "config.py").exists():
        PROJECT_ROOT = Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))

import importlib
if "config" in sys.modules:
    importlib.reload(sys.modules["config"])
from config import CFG
from src.utils.helpers import get_logger, set_seed

set_seed(CFG.RANDOM_SEED)
logger = get_logger("02_sentiment_analysis")

# ── Plot style (same palette as Notebook 01) ──────────────────────────────────
TICKER_COLORS = dict(zip(CFG.TICKERS, [
    "#1565C0", "#BF360C", "#2E7D32", "#6A1B9A", "#E65100"
]))
SENT_COLORS = {
    "positive": "#2E7D32",
    "neutral"  : "#90A4AE",
    "negative" : "#B71C1C",
}

plt.rcParams.update({
    "figure.dpi"        : 130,
    "figure.facecolor"  : "white",
    "axes.facecolor"    : "#F8F9FA",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.alpha"        : 0.3,
    "grid.linestyle"    : "--",
    "font.size"         : 11,
    "axes.titlesize"    : 13,
    "axes.titleweight"  : "bold",
    "axes.labelsize"    : 11,
    "legend.fontsize"   : 10,
})

print(f"Project root : {PROJECT_ROOT}")
print(f"Config v     : {CFG.VERSION}")
print(f"Device       : {CFG.DEVICE}")
print(f"Tickers      : {CFG.TICKERS}")


---
## 1 · Load Raw Data

Load all three text sources saved by Notebook 01 and print a quick inventory
so we can confirm what is available before processing begins.

In [ ]:
# =============================================================================
# CELL 03 | Load Raw Text Sources
#
# Loads all three raw text files produced by Notebook 01.
# Handles missing files gracefully — any source not collected yet
# will be represented as an empty DataFrame with the correct schema.
#
# Also sets USE_VADER flag:
#   False (default) = try FinBERT first, fall back to VADER on error
#   True            = skip FinBERT entirely, use fast VADER on CPU
#   Set to True if FinBERT is too slow on your machine.
# =============================================================================

# ── Scoring method control ────────────────────────────────────────────────────
# Change to True to use fast VADER instead of FinBERT on CPU
USE_VADER = False

# ── Load Alpha Vantage (pre-scored) ──────────────────────────────────────────
av_path = CFG.NEWS / "alphavantage_news.json"
if av_path.exists():
    av_raw = pd.read_json(av_path, orient="records")
    av_raw["published_at"] = pd.to_datetime(av_raw["published_at"], utc=True, errors="coerce")
    print(f"Alpha Vantage : {len(av_raw):,} articles | "
          f"{av_raw['published_at'].min().date()} -> {av_raw['published_at'].max().date()}")
else:
    av_raw = pd.DataFrame()
    print("Alpha Vantage : NOT FOUND — run Notebook 01 Cell 18b first")

# ── Load RSS News ─────────────────────────────────────────────────────────────
rss_path = CFG.NEWS / "reuters_articles.json"
if rss_path.exists():
    rss_raw = pd.read_json(rss_path, orient="records")
    rss_raw["published"] = pd.to_datetime(rss_raw["published"], utc=True, errors="coerce")
    print(f"RSS News      : {len(rss_raw):,} articles | "
          f"{rss_raw['published'].min().date()} -> {rss_raw['published'].max().date()}")
else:
    rss_raw = pd.DataFrame()
    print("RSS News      : NOT FOUND — run Notebook 01 Cell 08 first")

# ── Load NewsData.io ──────────────────────────────────────────────────────────
nd_path = CFG.TWITTER / "newsdata_articles.json"
if nd_path.exists():
    nd_raw = pd.read_json(nd_path, orient="records")
    nd_raw["published_at"] = pd.to_datetime(nd_raw["published_at"], utc=True, errors="coerce")
    print(f"NewsData.io   : {len(nd_raw):,} articles | "
          f"{nd_raw['published_at'].min().date()} -> {nd_raw['published_at'].max().date()}")
    print(f"              NOTE: dates are 2026 — will NOT align with price data (2021-2024)")
    print(f"              Scored for completeness but excluded from feature merge in NB03")
else:
    nd_raw = pd.DataFrame()
    print("NewsData.io   : NOT FOUND")

print(f"\nScoring method: {'VADER (fast, CPU)' if USE_VADER else 'FinBERT (accurate) with VADER fallback'}")
print(f"Total to score : {len(rss_raw) + len(nd_raw)} articles (AV pre-scored, skipped)")


---
## 2 · Alpha Vantage — Pre-Computed Sentiment

Alpha Vantage returns `av_sentiment_score` (ticker-specific, range [-1, +1])
and `relevance_score` (how relevant the article is to that ticker).

**We use these directly** — no NLP model needed.

The ticker-specific score is more precise than the article-level overall score
because it accounts for the fact that an article about the whole tech sector
may be bullish for NVDA but neutral for AAPL.

In [ ]:
# =============================================================================
# CELL 05 | Process Alpha Vantage Pre-Computed Scores
#
# Normalises the AV label vocabulary into the unified schema used
# across all sources in this notebook:
#   label  : 'positive' | 'neutral' | 'negative'
#   score  : float in [-1, +1]  (directly from av_sentiment_score)
#   method : 'alphavantage'
#
# We also apply a relevance filter: articles with relevance_score < 0.15
# are likely tangentially related to the ticker and may add noise.
# =============================================================================

AV_LABEL_MAP = {
    "Bullish"          : "positive",
    "Somewhat-Bullish" : "positive",
    "Neutral"          : "neutral",
    "Somewhat-Bearish" : "negative",
    "Bearish"          : "negative",
}

if not av_raw.empty:
    av_scored = av_raw.copy()

    # Apply relevance filter — keep only articles clearly relevant to the ticker
    before = len(av_scored)
    av_scored = av_scored[av_scored["relevance_score"] >= 0.15].reset_index(drop=True)
    print(f"Relevance filter (>= 0.15): {before} -> {len(av_scored)} articles "
          f"({before - len(av_scored)} low-relevance removed)")

    # Unified schema columns
    av_scored["label"]  = av_scored["av_sentiment_label"].map(AV_LABEL_MAP).fillna("neutral")
    av_scored["score"]  = av_scored["av_sentiment_score"].astype(float)
    av_scored["method"] = "alphavantage"
    av_scored["date"]   = av_scored["published_at"].dt.tz_localize(None).dt.normalize()
    av_scored["ticker"] = av_scored["ticker_ref"]

    # Sentiment label from score (as cross-check against AV label)
    av_scored["label_from_score"] = pd.cut(
        av_scored["score"],
        bins=[-1.1, -0.05, 0.05, 1.1],
        labels=["negative", "neutral", "positive"],
    )

    print(f"\nAlpha Vantage sentiment distribution:")
    print(av_scored["label"].value_counts().to_string())
    print(f"\nScore statistics:")
    print(av_scored["score"].describe().round(4).to_string())
    print(f"\nBy ticker:")
    print(av_scored.groupby("ticker")["score"].agg(["count","mean","std"]).round(3).to_string())
else:
    av_scored = pd.DataFrame()
    print("No Alpha Vantage data to process.")


In [ ]:
# =============================================================================
# CELL 06 | Plot — Alpha Vantage Sentiment Overview
#
# Three panels:
#   Left   : Score distribution histogram (all tickers combined).
#             Shows the natural tilt of financial news sentiment.
#   Centre : Mean sentiment score per ticker (bar chart).
#             Reveals which stocks receive more bullish vs bearish coverage.
#   Right  : Daily mean sentiment over time (line per ticker, 30-day rolling).
#             Shows how market sentiment evolved across the 2021-2024 window.
# =============================================================================

if not av_scored.empty:
    fig, (ax_hist, ax_ticker, ax_time) = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("Alpha Vantage — Pre-Computed Sentiment Scores (2021-2024)",
                 fontsize=14, fontweight="bold")

    # ── Left: Score distribution ──────────────────────────────────────────────
    ax_hist.hist(av_scored["score"], bins=40, color="#1565C0", alpha=0.82,
                 edgecolor="white")
    ax_hist.axvline(0,    color="#333", linewidth=1.0, linestyle="--", alpha=0.5)
    ax_hist.axvline(0.05, color="#2E7D32", linewidth=1.0, linestyle=":",
                    alpha=0.7, label="Positive threshold (+0.05)")
    ax_hist.axvline(-0.05,color="#B71C1C", linewidth=1.0, linestyle=":",
                    alpha=0.7, label="Negative threshold (-0.05)")
    ax_hist.set_title("Sentiment score distribution")
    ax_hist.set_xlabel("av_sentiment_score [-1, +1]")
    ax_hist.set_ylabel("Article count")
    ax_hist.legend(fontsize=8)

    # ── Centre: Mean score per ticker ─────────────────────────────────────────
    ticker_means = av_scored.groupby("ticker")["score"].mean().sort_values()
    colors = ["#B71C1C" if v < 0 else "#2E7D32" for v in ticker_means.values]
    bars = ax_ticker.barh(ticker_means.index, ticker_means.values,
                          color=colors, alpha=0.85, edgecolor="white", height=0.55)
    ax_ticker.axvline(0, color="#333", linewidth=0.8)
    for bar, val in zip(bars, ticker_means.values):
        ax_ticker.text(
            val + (0.003 if val >= 0 else -0.003),
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9,
            ha="left" if val >= 0 else "right",
        )
    ax_ticker.set_title("Mean sentiment score per ticker")
    ax_ticker.set_xlabel("Mean av_sentiment_score")

    # ── Right: 30-day rolling mean sentiment per ticker ───────────────────────
    av_scored_sorted = av_scored.sort_values("date")
    for ticker in CFG.TICKERS:
        t_df = (
            av_scored_sorted[av_scored_sorted["ticker"] == ticker]
            .set_index("date")["score"]
            .resample("D").mean()
            .rolling(30, min_periods=5)
            .mean()
        )
        if not t_df.empty:
            ax_time.plot(t_df.index, t_df.values,
                         label=ticker, color=TICKER_COLORS[ticker],
                         linewidth=1.6, alpha=0.85)

    ax_time.axhline(0, color="#333", linewidth=0.8, linestyle="--", alpha=0.5)
    ax_time.set_title("Daily sentiment (30-day rolling mean per ticker)")
    ax_time.set_ylabel("Mean sentiment score")
    ax_time.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax_time.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
    plt.setp(ax_time.xaxis.get_majorticklabels(), rotation=35, ha="right")
    ax_time.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(CFG.FIGURES / "02_av_sentiment_overview.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Saved -> {CFG.FIGURES / '02_av_sentiment_overview.png'}")


---
## 3 · FinBERT / VADER Scoring — RSS + NewsData.io

**FinBERT** (ProsusAI/finbert) is fine-tuned on financial text and significantly
outperforms general-purpose sentiment models on financial news.

**VADER** is a fast rule-based model requiring no GPU. It is less accurate on
financial text but runs in milliseconds per article on CPU.

**Decision logic in this notebook:**
- `USE_VADER = False` (default) → tries FinBERT, automatically falls back to VADER if it fails
- `USE_VADER = True` → skips FinBERT entirely, uses VADER directly

On local Mac CPU, FinBERT takes ~3-5 minutes for 277 articles.
On Colab A100, the same batch takes ~10 seconds.

> If you want to re-score on Colab later for better accuracy,
> set `USE_VADER = False` and re-run Cells 08-10 on Colab.

In [ ]:
# =============================================================================
# CELL 08 | Sentiment Scorer Setup
#
# Initialises whichever scoring backend is available and requested.
# The score() function provides a unified interface regardless of which
# backend is used — downstream cells don't need to know which one ran.
#
# Output schema from score():
#   pd.DataFrame with columns: text, label, score, confidence, method
#   label  : 'positive' | 'neutral' | 'negative'
#   score  : float [-1, +1]
# =============================================================================

FINBERT_LOADED = False
finbert_pipeline = None
vader_analyzer   = None


def load_finbert():
    """Loads FinBERT pipeline. Returns True if successful."""
    global finbert_pipeline, FINBERT_LOADED
    try:
        from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
        print("Loading FinBERT (ProsusAI/finbert) — first run downloads ~500MB ...")
        tok   = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        finbert_pipeline = pipeline(
            "text-classification",
            model=model, tokenizer=tok,
            device=-1,   # CPU — change to 0 on Colab GPU
            top_k=None,
        )
        FINBERT_LOADED = True
        print("FinBERT loaded successfully.")
        return True
    except Exception as exc:
        print(f"FinBERT load failed: {exc}")
        print("Falling back to VADER.")
        return False


def load_vader():
    """Loads VADER analyser."""
    global vader_analyzer
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    vader_analyzer = SentimentIntensityAnalyzer()
    print("VADER analyser ready.")


def score_finbert(texts: list, batch_size: int = 8) -> pd.DataFrame:
    """Scores texts with FinBERT. batch_size=8 is safe for CPU RAM."""
    rows = []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i : i + batch_size]
        outputs = finbert_pipeline(batch, truncation=True, max_length=512,
                                   padding=True, batch_size=len(batch))
        for text, output in zip(batch, outputs):
            best  = max(output, key=lambda x: x["score"])
            label = best["label"].lower()
            conf  = best["score"]
            p = next((x["score"] for x in output if x["label"].lower()=="positive"), 0.0)
            n = next((x["score"] for x in output if x["label"].lower()=="negative"), 0.0)
            rows.append({
                "text": text, "label": label,
                "score": float(p - n),   # [-1, +1]
                "confidence": conf, "method": "finbert",
            })
        if (i // batch_size) % 5 == 0:
            print(f"  Scored {min(i+batch_size, len(texts))}/{len(texts)} ...", end="\r")
    return pd.DataFrame(rows)


def score_vader(texts: list) -> pd.DataFrame:
    """Scores texts with VADER."""
    rows = []
    for text in texts:
        scores   = vader_analyzer.polarity_scores(str(text))
        compound = scores["compound"]
        label    = ("positive" if compound >= 0.05
                    else "negative" if compound <= -0.05
                    else "neutral")
        rows.append({
            "text": text, "label": label,
            "score": compound, "confidence": abs(compound),
            "method": "vader",
        })
    return pd.DataFrame(rows)


def score(texts: list, batch_size: int = 8) -> pd.DataFrame:
    """
    Unified scoring function — uses FinBERT or VADER depending on
    USE_VADER flag and what loaded successfully.
    """
    if not texts:
        return pd.DataFrame(columns=["text","label","score","confidence","method"])

    if FINBERT_LOADED and not USE_VADER:
        return score_finbert(texts, batch_size)
    else:
        return score_vader(texts)


# ── Initialise the chosen backend ─────────────────────────────────────────────
print(f"USE_VADER = {USE_VADER}")
if USE_VADER:
    load_vader()
    METHOD = "vader"
else:
    success = load_finbert()
    if not success:
        load_vader()
    METHOD = "finbert" if FINBERT_LOADED else "vader"

print(f"\nActive scoring method: {METHOD.upper()}")


In [ ]:
# =============================================================================
# CELL 09 | Score RSS and NewsData.io Articles
#
# Runs RSS (77) and NewsData.io (200) articles through the active scorer.
# Total: ~277 articles.
# FinBERT CPU estimate: ~3-5 minutes.
# VADER estimate: ~2 seconds.
#
# RSS articles have no ticker_ref — they cover general market news.
# We keep them without a ticker and broadcast them to all tickers
# during the daily aggregation step in Cell 12.
# =============================================================================

rss_scored = pd.DataFrame()
nd_scored  = pd.DataFrame()

# ── Score RSS articles ────────────────────────────────────────────────────────
if not rss_raw.empty:
    print(f"Scoring {len(rss_raw)} RSS articles with {METHOD.upper()} ...")
    t0 = time.time()

    texts       = rss_raw["text"].fillna("").tolist()
    score_df    = score(texts)

    rss_scored  = rss_raw.copy()
    rss_scored["label"]      = score_df["label"].values
    rss_scored["score"]      = score_df["score"].values
    rss_scored["confidence"] = score_df["confidence"].values
    rss_scored["method"]     = score_df["method"].values
    rss_scored["date"]       = rss_scored["published"].dt.tz_localize(None).dt.normalize()
    rss_scored["ticker"]     = "ALL"   # broadcast to all tickers during aggregation

    elapsed = time.time() - t0
    print(f"\nRSS scoring complete in {elapsed:.1f}s")
    print(f"Label distribution: {rss_scored['label'].value_counts().to_dict()}")
    print(f"Mean score: {rss_scored['score'].mean():.4f}")
else:
    print("No RSS articles to score.")

# ── Score NewsData.io articles ────────────────────────────────────────────────
if not nd_raw.empty:
    print(f"\nScoring {len(nd_raw)} NewsData.io articles with {METHOD.upper()} ...")
    t0 = time.time()

    texts    = nd_raw["text"].fillna("").tolist()
    score_df = score(texts)

    nd_scored = nd_raw.copy()
    nd_scored["label"]      = score_df["label"].values
    nd_scored["score"]      = score_df["score"].values
    nd_scored["confidence"] = score_df["confidence"].values
    nd_scored["method"]     = score_df["method"].values
    nd_scored["date"]       = nd_scored["published_at"].dt.tz_localize(None).dt.normalize()
    nd_scored["ticker"]     = nd_scored["ticker_ref"]

    elapsed = time.time() - t0
    print(f"\nNewsData scoring complete in {elapsed:.1f}s")
    print(f"Label distribution: {nd_scored['label'].value_counts().to_dict()}")
    print(f"NOTE: Dates are 2026 — excluded from feature merge in Notebook 03")
else:
    print("No NewsData.io articles to score.")


In [ ]:
# =============================================================================
# CELL 10 | Plot — FinBERT/VADER Scoring Results
#
# Two panels comparing the scored RSS and NewsData distributions.
# This validates the scorer is working correctly — financial news should
# skew slightly positive (markets trend upward long-term) with a dominant
# neutral class (most news is factual, not opinion-driven).
# =============================================================================

sources_to_plot = {}
if not rss_scored.empty:
    sources_to_plot["RSS News"] = rss_scored
if not nd_scored.empty:
    sources_to_plot["NewsData.io"] = nd_scored

if sources_to_plot:
    n = len(sources_to_plot)
    fig, axes = plt.subplots(1, n * 2, figsize=(7 * n, 5))
    if n == 1:
        axes = list(axes)
    fig.suptitle(f"FinBERT/VADER Scoring Results — RSS & NewsData.io",
                 fontsize=14, fontweight="bold")

    ax_pairs = [(axes[i*2], axes[i*2+1]) for i in range(n)]

    for (name, df), (ax_label, ax_hist) in zip(sources_to_plot.items(), ax_pairs):
        # Label distribution bar
        counts = df["label"].value_counts()
        bar_colors = [SENT_COLORS.get(l, "#888") for l in counts.index]
        bars = ax_label.bar(counts.index, counts.values, color=bar_colors,
                            width=0.45, edgecolor="white")
        for bar, val in zip(bars, counts.values):
            pct = val / len(df) * 100
            ax_label.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.3,
                          f"{val}\n({pct:.0f}%)", ha="center", va="bottom", fontsize=10)
        ax_label.set_title(f"{name}\nSentiment labels ({df['method'].iloc[0]})")
        ax_label.set_ylabel("Article count")

        # Score histogram
        ax_hist.hist(df["score"], bins=25, color="#1565C0", alpha=0.82, edgecolor="white")
        ax_hist.axvline(0,     color="#333",    linewidth=1.0, linestyle="--", alpha=0.5)
        ax_hist.axvline(0.05,  color="#2E7D32", linewidth=1.0, linestyle=":", alpha=0.7)
        ax_hist.axvline(-0.05, color="#B71C1C", linewidth=1.0, linestyle=":", alpha=0.7)
        ax_hist.set_title(f"{name}\nScore distribution")
        ax_hist.set_xlabel("Sentiment score [-1, +1]")

    plt.tight_layout()
    plt.savefig(CFG.FIGURES / "02_scored_distributions.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Saved -> {CFG.FIGURES / '02_scored_distributions.png'}")


---
## 4 · Combine All Sources

Merge Alpha Vantage, RSS, and NewsData.io into a single unified DataFrame
with a consistent schema before aggregating to daily features.

**Handling RSS (no ticker):**
RSS articles cover general market news with no specific ticker.
We create one copy per ticker (broadcast) so general market sentiment
contributes to every ticker's daily feature equally.

**Handling NewsData.io (2026 dates):**
Kept in the unified DataFrame but will naturally produce zero overlap
with the 2021-2024 price data. Notebook 03's date merge will exclude them automatically.

In [ ]:
# =============================================================================
# CELL 12 | Combine All Scored Sources
#
# Builds a unified DataFrame with consistent columns across all sources:
#   date   : datetime (no timezone, normalised to midnight)
#   ticker : ticker symbol
#   score  : float [-1, +1]
#   label  : 'positive' | 'neutral' | 'negative'
#   source : 'alphavantage' | 'rss' | 'newsdata'
#   method : 'alphavantage' | 'finbert' | 'vader'
#
# RSS broadcast: one row per (article, ticker) — 77 articles × 5 tickers
# = 385 rows added from RSS. This is correct behaviour — general market
# news is relevant to all tracked stocks.
# =============================================================================

UNIFIED_COLS = ["date", "ticker", "score", "label", "source", "method"]
parts = []

# ── Alpha Vantage ─────────────────────────────────────────────────────────────
if not av_scored.empty:
    av_part = av_scored[["date", "ticker", "score", "label", "method"]].copy()
    av_part["source"] = "alphavantage"
    parts.append(av_part[UNIFIED_COLS])
    print(f"Alpha Vantage : {len(av_part):,} rows added")

# ── RSS (broadcast to all tickers) ───────────────────────────────────────────
if not rss_scored.empty:
    rss_rows = []
    for ticker in CFG.TICKERS:
        t_df = rss_scored[["date","score","label","method"]].copy()
        t_df["ticker"] = ticker
        t_df["source"] = "rss"
        rss_rows.append(t_df[UNIFIED_COLS])
    rss_part = pd.concat(rss_rows, ignore_index=True)
    parts.append(rss_part)
    print(f"RSS News      : {len(rss_scored):,} articles x {len(CFG.TICKERS)} tickers "
          f"= {len(rss_part):,} rows added")

# ── NewsData.io (ticker-specific, 2026) ───────────────────────────────────────
if not nd_scored.empty:
    nd_part = nd_scored[nd_scored["ticker"].isin(CFG.TICKERS)].copy()
    nd_part = nd_part[["date","ticker","score","label","method"]].copy()
    nd_part["source"] = "newsdata"
    parts.append(nd_part[UNIFIED_COLS])
    print(f"NewsData.io   : {len(nd_part):,} rows added (2026 dates — excluded in NB03 merge)")

# ── Combine ───────────────────────────────────────────────────────────────────
if parts:
    unified_df = pd.concat(parts, ignore_index=True)
    unified_df["date"]   = pd.to_datetime(unified_df["date"])
    unified_df["score"]  = unified_df["score"].astype(float)
    unified_df = unified_df.dropna(subset=["date","score"]).reset_index(drop=True)

    print(f"\nUnified DataFrame: {len(unified_df):,} total rows")
    print(f"Date range       : {unified_df['date'].min().date()} -> {unified_df['date'].max().date()}")
    print(f"\nRows per source:")
    print(unified_df["source"].value_counts().to_string())
    print(f"\nRows per ticker:")
    print(unified_df["ticker"].value_counts().to_string())
else:
    unified_df = pd.DataFrame(columns=UNIFIED_COLS)
    print("WARNING: No scored data available. Check that Notebook 01 ran successfully.")


In [ ]:
# =============================================================================
# CELL 13 | Plot — Combined Sentiment Overview
#
# Two panels showing the full unified dataset:
#   Left  : Source breakdown (stacked bar by sentiment label per source).
#           Reveals whether different sources have different sentiment biases.
#   Right : Articles per week across the full timeline (stacked by source).
#           Shows the temporal density of our combined sentiment data.
# =============================================================================

if not unified_df.empty:
    fig, (ax_source, ax_time) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Combined Sentiment Dataset Overview", fontsize=14, fontweight="bold")

    # ── Left: Stacked label distribution per source ───────────────────────────
    src_label = (
        unified_df.groupby(["source","label"]).size().unstack(fill_value=0)
    )
    # Ensure all label columns exist
    for lbl in ["positive","neutral","negative"]:
        if lbl not in src_label.columns:
            src_label[lbl] = 0

    x      = np.arange(len(src_label))
    width  = 0.5
    bottom = np.zeros(len(src_label))

    for lbl in ["positive","neutral","negative"]:
        ax_source.bar(x, src_label[lbl], width, bottom=bottom,
                      label=lbl.capitalize(), color=SENT_COLORS[lbl],
                      alpha=0.85, edgecolor="white")
        bottom += src_label[lbl].values

    ax_source.set_xticks(x)
    ax_source.set_xticklabels(src_label.index, rotation=15, ha="right")
    ax_source.set_title("Sentiment labels by source")
    ax_source.set_ylabel("Article count")
    ax_source.legend()

    # ── Right: Weekly article count by source ─────────────────────────────────
    src_colors = {
        "alphavantage": "#1565C0",
        "rss"         : "#B71C1C",
        "newsdata"    : "#E65100",
    }
    weekly = (
        unified_df[unified_df["date"].dt.year <= 2024]   # exclude 2026 newsdata
        .assign(week=lambda d: d["date"].dt.to_period("W").dt.start_time)
        .groupby(["week","source"]).size().unstack(fill_value=0)
    )

    if not weekly.empty:
        bottom_w = np.zeros(len(weekly))
        for src in weekly.columns:
            ax_time.bar(weekly.index, weekly[src], width=5,
                        bottom=bottom_w, label=src,
                        color=src_colors.get(src,"#888"), alpha=0.82, edgecolor="none")
            bottom_w += weekly[src].values

        ax_time.set_title("Weekly article volume by source (2021-2024 only)")
        ax_time.set_ylabel("Articles per week")
        ax_time.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax_time.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
        plt.setp(ax_time.xaxis.get_majorticklabels(), rotation=35, ha="right")
        ax_time.legend()

    plt.tight_layout()
    plt.savefig(CFG.FIGURES / "02_combined_overview.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Saved -> {CFG.FIGURES / '02_combined_overview.png'}")


---
## 5 · Daily Sentiment Aggregation

Collapse article-level scores into **one row per (trading day, ticker)**.

**Aggregation columns produced:**

| Column | Description |
|---|---|
| `sentiment_mean` | Mean score across all sources that day |
| `sentiment_std` | Std of scores — high std = conflicting signals |
| `pos_ratio` | Fraction of articles with positive label |
| `neg_ratio` | Fraction of articles with negative label |
| `record_count` | Total articles contributing to that day |
| `av_sentiment_mean` | Alpha Vantage score only (most precise) |
| `av_record_count` | Alpha Vantage article count that day |

**Forward-fill strategy:**
Gaps (trading days with no articles) are forward-filled up to 5 days.
Beyond 5 days the sentiment is set to 0 (neutral) — far enough back
that the signal has decayed.

In [ ]:
# =============================================================================
# CELL 15 | Daily Sentiment Aggregation + Forward Fill
#
# Step 1: Aggregate all sources to (date, ticker) level
# Step 2: Build a complete date spine (all trading days from price data)
# Step 3: Left-join sentiment onto trading days
# Step 4: Forward-fill gaps up to 5 days, then fill remaining with 0
#
# The output is a DataFrame with one row per (trading day, ticker) that
# Notebook 03 can directly merge with the price + technical feature DataFrame.
# =============================================================================

if unified_df.empty:
    print("No unified data — skipping aggregation.")
    sentiment_daily = pd.DataFrame()
else:
    # ── Step 1: Aggregate to (date, ticker) ──────────────────────────────────
    # Only include 2021-2024 data (exclude 2026 NewsData dates)
    unified_train = unified_df[
        (unified_df["date"] >= CFG.HIST_START) &
        (unified_df["date"] <= CFG.HIST_END)
    ].copy()

    def agg_sentiment(group):
        """Aggregates one (date, ticker) group into a single row."""
        scores = group["score"].values
        labels = group["label"].values
        return pd.Series({
            "sentiment_mean": float(np.mean(scores)),
            "sentiment_std" : float(np.std(scores)) if len(scores) > 1 else 0.0,
            "pos_ratio"     : float(np.mean(labels == "positive")),
            "neg_ratio"     : float(np.mean(labels == "negative")),
            "record_count"  : len(scores),
        })

    # All sources combined
    daily_all = (
        unified_train
        .groupby(["date","ticker"])
        .apply(agg_sentiment)
        .reset_index()
    )

    # Alpha Vantage only (ticker-specific, more precise)
    av_only = unified_train[unified_train["source"] == "alphavantage"]
    if not av_only.empty:
        daily_av = (
            av_only
            .groupby(["date","ticker"])
            .agg(av_sentiment_mean=("score","mean"),
                 av_record_count=("score","count"))
            .reset_index()
        )
        daily_all = daily_all.merge(daily_av, on=["date","ticker"], how="left")
    else:
        daily_all["av_sentiment_mean"] = np.nan
        daily_all["av_record_count"]   = 0

    daily_all["av_record_count"] = daily_all["av_record_count"].fillna(0).astype(int)

    print(f"Aggregated: {len(daily_all):,} (date, ticker) pairs")
    print(f"Date range: {daily_all['date'].min().date()} -> {daily_all['date'].max().date()}")

    # ── Step 2: Build complete trading day spine ──────────────────────────────
    # Load price data to get exact trading days (excludes weekends + holidays)
    price_csv = CFG.PRICES / f"{CFG.TICKERS[0]}_daily.csv"
    if price_csv.exists():
        price_ref = pd.read_csv(price_csv, parse_dates=["date"])
        trading_days = price_ref["date"].dt.normalize().unique()
        print(f"\nTrading days in price data: {len(trading_days):,}")
    else:
        # Fallback: all calendar days in range
        trading_days = pd.date_range(CFG.HIST_START, CFG.HIST_END, freq="B")
        print(f"\nUsing business days (price CSV not found): {len(trading_days):,}")

    # Full spine: every (trading_day, ticker) combination
    spine = pd.DataFrame([
        {"date": pd.Timestamp(day), "ticker": ticker}
        for day in trading_days
        for ticker in CFG.TICKERS
    ])
    spine["date"] = pd.to_datetime(spine["date"])
    print(f"Full spine (trading days x tickers): {len(spine):,} rows")

    # ── Step 3: Left-join sentiment onto spine ────────────────────────────────
    daily_all["date"] = pd.to_datetime(daily_all["date"])
    sentiment_daily   = spine.merge(daily_all, on=["date","ticker"], how="left")

    before_fill = sentiment_daily["sentiment_mean"].notna().sum()
    print(f"\nRows with sentiment data before fill: {before_fill:,} / {len(sentiment_daily):,} "
          f"({before_fill/len(sentiment_daily)*100:.1f}%)")

    # ── Step 4: Forward-fill gaps (per ticker) ────────────────────────────────
    sent_cols = ["sentiment_mean","sentiment_std","pos_ratio","neg_ratio",
                 "record_count","av_sentiment_mean","av_record_count"]

    sentiment_daily = (
        sentiment_daily
        .sort_values(["ticker","date"])
        .groupby("ticker", group_keys=False)
        .apply(lambda g: g.fillna(method="ffill", limit=5))   # forward-fill up to 5 days
        .reset_index(drop=True)
    )

    # Fill remaining NaN (gaps > 5 days) with neutral (0)
    sentiment_daily["sentiment_mean"]    = sentiment_daily["sentiment_mean"].fillna(0.0)
    sentiment_daily["sentiment_std"]     = sentiment_daily["sentiment_std"].fillna(0.0)
    sentiment_daily["pos_ratio"]         = sentiment_daily["pos_ratio"].fillna(0.0)
    sentiment_daily["neg_ratio"]         = sentiment_daily["neg_ratio"].fillna(0.0)
    sentiment_daily["record_count"]      = sentiment_daily["record_count"].fillna(0).astype(int)
    sentiment_daily["av_sentiment_mean"] = sentiment_daily["av_sentiment_mean"].fillna(0.0)
    sentiment_daily["av_record_count"]   = sentiment_daily["av_record_count"].fillna(0).astype(int)

    after_fill = (sentiment_daily["record_count"] > 0).sum()
    print(f"Rows with actual data after fill  : {after_fill:,} / {len(sentiment_daily):,} "
          f"({after_fill/len(sentiment_daily)*100:.1f}%)")

    print(f"\nFinal sentiment_daily shape: {sentiment_daily.shape}")
    print(f"Columns: {list(sentiment_daily.columns)}")
    print(f"\nSample (AAPL first 5 rows with data):")
    sample = sentiment_daily[
        (sentiment_daily["ticker"]=="AAPL") &
        (sentiment_daily["record_count"]>0)
    ].head(5)
    print(sample[["date","ticker","sentiment_mean","av_sentiment_mean","record_count"]].to_string())


In [ ]:
# =============================================================================
# CELL 16 | Plot — Daily Sentiment Time Series
#
# Two panels:
#   Top  : Daily mean sentiment per ticker (7-day rolling average).
#          This is the actual feature vector that will feed the models.
#          Light grey regions = days filled with forward-fill or neutral.
#   Bottom: Record count per day (stacked by ticker).
#          Shows where we have strong signal vs imputed values.
# =============================================================================

if not sentiment_daily.empty:
    fig, (ax_sent, ax_count) = plt.subplots(2, 1, figsize=(15, 8),
                                             sharex=True, height_ratios=[3,1])
    fig.suptitle("Daily Sentiment Features — Ready for Notebook 03",
                 fontsize=14, fontweight="bold")

    # ── Top: Rolling sentiment per ticker ─────────────────────────────────────
    for ticker in CFG.TICKERS:
        t_df = sentiment_daily[sentiment_daily["ticker"]==ticker].sort_values("date")
        rolled = t_df.set_index("date")["sentiment_mean"].rolling(7, min_periods=1).mean()
        ax_sent.plot(rolled.index, rolled.values,
                     label=ticker, color=TICKER_COLORS[ticker],
                     linewidth=1.6, alpha=0.85)

    ax_sent.axhline(0, color="#333", linewidth=0.8, linestyle="--", alpha=0.4)
    ax_sent.axhspan(-0.05, 0.05, alpha=0.04, color="#90A4AE")   # neutral band
    ax_sent.set_ylabel("Sentiment score (7-day rolling mean)")
    ax_sent.set_title("Daily sentiment per ticker — this is the feature fed to RNN/LSTM/GRU")
    ax_sent.legend(ncols=5, fontsize=9)
    ax_sent.set_ylim(-0.6, 0.6)

    # ── Bottom: Record count ──────────────────────────────────────────────────
    # Use AAPL as representative (others similar due to RSS broadcast)
    aapl = sentiment_daily[sentiment_daily["ticker"]=="AAPL"].sort_values("date")
    # Split into actual vs imputed
    has_data = aapl["record_count"] > 0
    ax_count.bar(aapl[has_data]["date"],  aapl[has_data]["record_count"],
                 width=1, color="#1565C0", alpha=0.7, label="Actual articles")
    ax_count.bar(aapl[~has_data]["date"], np.ones((~has_data).sum()),
                 width=1, color="#E0E0E0", alpha=0.5, label="Forward-filled / neutral")

    ax_count.set_ylabel("Articles/day (AAPL)")
    ax_count.set_title("Data density — blue = real signal, grey = imputed")
    ax_count.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax_count.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
    plt.setp(ax_count.xaxis.get_majorticklabels(), rotation=35, ha="right")
    ax_count.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(CFG.FIGURES / "02_daily_sentiment_features.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Saved -> {CFG.FIGURES / '02_daily_sentiment_features.png'}")


---
## 6 · Save — sentiment_daily.parquet

The final output of this notebook is a single Parquet file:
`data/processed/sentiment_daily.parquet`

Parquet is used instead of CSV because:
- Preserves dtypes (datetime, float32) without re-parsing on load
- ~5x smaller file size than CSV for this schema
- Loads in milliseconds in Notebook 03

A CSV copy is also saved for human inspection.

In [ ]:
# =============================================================================
# CELL 18 | Save — bypass pandas to_csv entirely, use Python csv module
# =============================================================================
import csv

if not sentiment_daily.empty:

    csv_path = CFG.DATA_PROC / "sentiment_daily.csv"

    # Extract every column as a raw Python list — no pandas involved in write
    dates      = pd.to_datetime(sentiment_daily["date"].values)
    tickers    = sentiment_daily["ticker"].tolist()
    s_mean     = sentiment_daily["sentiment_mean"].tolist()
    s_std      = sentiment_daily["sentiment_std"].tolist()
    pos_ratio  = sentiment_daily["pos_ratio"].tolist()
    neg_ratio  = sentiment_daily["neg_ratio"].tolist()
    rec_count  = sentiment_daily["record_count"].tolist()
    av_mean    = sentiment_daily["av_sentiment_mean"].tolist()
    av_count   = sentiment_daily["av_record_count"].tolist()

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        # Header
        writer.writerow([
            "date","ticker","sentiment_mean","sentiment_std",
            "pos_ratio","neg_ratio","record_count",
            "av_sentiment_mean","av_record_count"
        ])
        # Rows
        for i in range(len(tickers)):
            writer.writerow([
                str(dates[i])[:10],   # YYYY-MM-DD
                str(tickers[i]),
                round(float(s_mean[i]),   6),
                round(float(s_std[i]),    6),
                round(float(pos_ratio[i]),6),
                round(float(neg_ratio[i]),6),
                int(rec_count[i]),
                round(float(av_mean[i]),  6),
                int(av_count[i]),
            ])

    print(f"Saved : {csv_path}")
    print(f"Size  : {csv_path.stat().st_size / 1024:.1f} KB")
    print(f"Rows  : {len(tickers):,}")

    # ── Coverage per ticker ───────────────────────────────────────────────────
    print(f"\nCoverage per ticker:")
    for ticker in CFG.TICKERS:
        idxs          = [i for i,t in enumerate(tickers) if t == ticker]
        days_total    = len(idxs)
        days_with_data= sum(1 for i in idxs if int(rec_count[i]) > 0)
        mean_sent     = sum(float(s_mean[i]) for i in idxs) / days_total if days_total else 0
        av_vals       = [float(av_mean[i]) for i in idxs if float(av_mean[i]) != 0]
        mean_av       = sum(av_vals) / len(av_vals) if av_vals else 0.0
        pct           = days_with_data / days_total * 100 if days_total else 0
        print(f"  {ticker:<5} | {days_total:>4} days | "
              f"{days_with_data:>4} with data ({pct:.1f}%) | "
              f"mean_sent={mean_sent:.4f} | mean_av={mean_av:.4f}")

    print(f"\n{'='*55}")
    print(f"  Notebook 02 complete.")
    print(f"  Output : data/processed/sentiment_daily.csv")
    print(f"  Next   : 03_feature_engineering.ipynb")
    print(f"{'='*55}")

else:
    print("WARNING: sentiment_daily is empty.")